In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Φορτώνουμε embeddings και labels
models_path = os.path.expanduser("~/igbert-embedding-analysis/results/models/")
figures_path = os.path.expanduser("~/igbert-embedding-analysis/results/figures/")
data_path = os.path.expanduser("~/igbert-embedding-analysis/data/processed/sequences_balanced.csv")

embeddings_l30 = np.load(models_path + "embeddings_l30.npy")
embeddings_l15 = np.load(models_path + "embeddings_l15.npy")
df = pd.read_csv(data_path)

print(f"Embeddings: {embeddings_l30.shape}")
print(f"Dataset: {df.shape}")
print(f"\nΣτήλες: {df.columns.tolist()}")

Embeddings: (4500, 1024)
Dataset: (4500, 8)

Στήλες: ['sequence_alignment_aa', 'v_call', 'j_call', 'junction_aa_length', 'v_identity', 'v_family', 'j_family', 'isotype']


In [2]:
def find_nearest_neighbors(query_idx, embeddings, n_neighbors=5):
    """
    Για μια αλληλουχία (query_idx), βρίσκει τις n πιο παρόμοιες
    στον embedding χώρο χρησιμοποιώντας cosine similarity.
    """
    # Υπολογίζουμε cosine similarity μεταξύ της query και όλων των άλλων
    query_embedding = embeddings[query_idx].reshape(1, -1)
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    # Αποκλείουμε την ίδια την αλληλουχία (similarity=1.0 με τον εαυτό της)
    similarities[query_idx] = -1
    
    # Παίρνουμε τους n πιο κοντινούς
    neighbor_indices = np.argsort(similarities)[::-1][:n_neighbors]
    neighbor_similarities = similarities[neighbor_indices]
    
    return neighbor_indices, neighbor_similarities

def show_neighbors(query_idx, embeddings, df, n_neighbors=5):
    """
    Δείχνει την query αλληλουχία και τους nearest neighbors της.
    """
    neighbor_indices, neighbor_similarities = find_nearest_neighbors(
        query_idx, embeddings, n_neighbors
    )
    
    query = df.iloc[query_idx]
    print(f"QUERY αλληλουχία #{query_idx}:")
    print(f"  V-gene: {query['v_family']} | J-gene: {query['j_family']} | "
          f"Isotype: {query['isotype']} | CDR3 length: {query['junction_aa_length']:.0f}")
    print(f"\n{n_neighbors} Nearest Neighbors:")
    print(f"{'#':<5} {'Similarity':<12} {'V-gene':<10} {'J-gene':<10} "
          f"{'Isotype':<8} {'CDR3 len':<10} {'Same V?'}")
    print("-" * 65)
    
    for rank, (idx, sim) in enumerate(zip(neighbor_indices, neighbor_similarities)):
        neighbor = df.iloc[idx]
        same_v = "✓" if neighbor['v_family'] == query['v_family'] else "✗"
        print(f"{rank+1:<5} {sim:<12.4f} {neighbor['v_family']:<10} "
              f"{neighbor['j_family']:<10} {neighbor['isotype']:<8} "
              f"{neighbor['junction_aa_length']:<10.0f} {same_v}")

# Δοκιμάζουμε με 3 τυχαίες αλληλουχίες
for idx in [0, 1500, 3000]:
    show_neighbors(idx, embeddings_l30, df)
    print()

QUERY αλληλουχία #0:
  V-gene: IGHV1 | J-gene: IGHJ4 | Isotype: IGHA | CDR3 length: 21

5 Nearest Neighbors:
#     Similarity   V-gene     J-gene     Isotype  CDR3 len   Same V?
-----------------------------------------------------------------
1     0.9978       IGHV1      IGHJ4      IGHA     21         ✓
2     0.9975       IGHV1      IGHJ4      IGHA     21         ✓
3     0.9973       IGHV1      IGHJ4      IGHA     21         ✓
4     0.9970       IGHV1      IGHJ4      IGHA     21         ✓
5     0.9965       IGHV1      IGHJ4      IGHA     21         ✓

QUERY αλληλουχία #1500:
  V-gene: IGHV3 | J-gene: IGHJ6 | Isotype: IGHA | CDR3 length: 23

5 Nearest Neighbors:
#     Similarity   V-gene     J-gene     Isotype  CDR3 len   Same V?
-----------------------------------------------------------------
1     0.9954       IGHV3      IGHJ6      IGHA     23         ✓
2     0.9923       IGHV3      IGHJ6      IGHA     23         ✓
3     0.9902       IGHV3      IGHJ6      IGHA     23         ✓
4   

In [ ]:
# Υπολογίζουμε για όλες τις αλληλουχίες τι % των neighbors έχουν ίδιο V-gene, isotype κλπ

same_v_counts = []
same_j_counts = []
same_iso_counts = []
same_cdr3_counts = []

print("Υπολογίζω nearest neighbors για όλες τις αλληλουχίες...")
for idx in range(len(df)):
    if idx % 500 == 0:
        print(f"  {idx}/4500...")
    
    neighbor_indices, _ = find_nearest_neighbors(idx, embeddings_l30, n_neighbors=5)
    
    query = df.iloc[idx]
    neighbors = df.iloc[neighbor_indices]
    
    same_v = (neighbors['v_family'] == query['v_family']).mean()
    same_j = (neighbors['j_family'] == query['j_family']).mean()
    same_iso = (neighbors['isotype'] == query['isotype']).mean()
    same_cdr3 = (abs(neighbors['junction_aa_length'] - query['junction_aa_length']) <= 2).mean()
    
    same_v_counts.append(same_v)
    same_j_counts.append(same_j)
    same_iso_counts.append(same_iso)
    same_cdr3_counts.append(same_cdr3)

print(f"\nΑποτελέσματα (μέσο % neighbors με ίδια ιδιότητα):")
print(f"V-gene family:  {np.mean(same_v_counts)*100:.1f}%")
print(f"J-gene family:  {np.mean(same_j_counts)*100:.1f}%")
print(f"Isotype:        {np.mean(same_iso_counts)*100:.1f}%")
print(f"CDR3 length±2:  {np.mean(same_cdr3_counts)*100:.1f}%")
print(f"\nRandom baseline: {100/3:.1f}%")

Υπολογίζω nearest neighbors για όλες τις αλληλουχίες...
  0/4500...
  500/4500...
  1000/4500...
  1500/4500...
  2000/4500...
  2500/4500...
  3000/4500...
  3500/4500...
  4000/4500...
